In [1]:
from pyspark.sql.functions import col, trim, lower, upper, regexp_replace, to_date, to_timestamp
import os
from pyspark.sql import SparkSession

os.environ["AWS_REGION"] = "us-east-1"
os.environ["AWS_ACCESS_KEY_ID"] = "admin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "password"

iceberg_version = "1.4.3"
aws_version = "3.3.4"
packages = [
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{iceberg_version}",
    f"org.apache.iceberg:iceberg-aws-bundle:{iceberg_version}",
    f"org.apache.hadoop:hadoop-aws:{aws_version}",
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
]

spark = SparkSession.builder \
    .appName("Ingestao-CGU-Favorecidos-PJ-Bronze") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.iceberg.type", "hive") \
    .config("spark.sql.catalog.iceberg.uri", "thrift://hive-metastore:9083") \
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.iceberg.warehouse", "s3a://warehouse/") \
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.iceberg.client.region", "us-east-1") \
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Lendo da camada Bronze (CNPJ Bruto)...")
df_bronze = spark.table("iceberg.bronze.favorecidos_pj")

print("Aplicando limpeza + tipagem (Camada Prata)...")
df_silver = (
    df_bronze
    .select(
        col("CNPJ").alias("cnpj"),
        col("RAZAOSOCIAL").alias("razao_social"),
        col("NOMEFANTASIA").alias("nome_fantasia"),
        col("COD_CNAE").alias("cod_cnae"),
        col("COD_NATJURIDICA").alias("cod_natjuridica"),
        col("TIPO_PESSOA").alias("tipo_pessoa"),
        col("LOGRADOURO").alias("logradouro"),
        col("NUMERO").alias("numero"),
        col("COMPLEMENTO").alias("complemento"),
        col("CEP").alias("cep"),
        col("BAIRRO").alias("bairro"),
        col("MUNICIPIO").alias("municipio"),
        col("UF").alias("uf"),
        col("data_ingestao").alias("data_ingestao")
    )
    .withColumn("cnpj", regexp_replace(trim(col("cnpj")), "[^0-9]", ""))
    .withColumn("razao_social", upper(trim(col("razao_social"))))
    .withColumn("nome_fantasia", upper(trim(col("nome_fantasia"))))
    .withColumn("tipo_pessoa", upper(trim(col("tipo_pessoa"))))
    .withColumn("logradouro", trim(col("logradouro")))
    .withColumn("numero", trim(col("numero")))
    .withColumn("complemento", trim(col("complemento")))
    .withColumn("cep", regexp_replace(trim(col("cep")), "[^0-9]", ""))
    .withColumn("bairro", upper(trim(col("bairro"))))
    .withColumn("municipio", upper(trim(col("municipio"))))
    .withColumn("uf", upper(trim(col("uf"))))
    .withColumn("cod_cnae", col("cod_cnae").cast("int"))
    .withColumn("cod_natjuridica", col("cod_natjuridica").cast("int"))
    .withColumn("data_ingestao", to_timestamp(col("data_ingestao")))
    .filter(col("cnpj").isNotNull() & (col("cnpj") != "") & (col("cnpj").rlike("^[0-9]{14}$")))
    .filter(col("razao_social").isNotNull() & (col("razao_social") != ""))
    .filter(col("uf").isNotNull() & (col("uf").rlike("^[A-Z]{2}$")))
)

print("Gravando na camada Prata...")
df_silver.writeTo("iceberg.silver.cnpj_cleansed") \
    .tableProperty("format-version", "2") \
    .using("iceberg") \
    .createOrReplace()

print("Prata finalizada com limpeza e tipagem.")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/vscode/.ivy2/cache
The jars for the packages stored in: /home/vscode/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-078aae15-8fe2-4b0d-b297-cac17a9dae17;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.4.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 263ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [defaul

Lendo da camada Bronze (CNPJ Bruto)...
Aplicando limpeza + tipagem (Camada Prata)...
Gravando na camada Prata...


Prata finalizada com limpeza e tipagem.


In [ ]:
df_show = spark.table("iceberg.silver.cnpj_cleansed").show(100, truncate=False)